In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np
from aind_behavior_gym.dynamic_foraging.task import CoupledBlockTask, UncoupledBlockTask
from aind_dynamic_foraging_models.generative_model import ForagerCollection


from aind_dynamic_foraging_models.generative_model.foragers import ForagerCollection


forager_collection: ForagerCollection = ForagerCollection()
df = forager_collection.get_all_foragers()

df


In [ ]:
from __future__ import annotations

import copy
import glob
import json
import os
import traceback
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Set

import numpy as np

# Ensure non-interactive backend (no display)
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from pynwb import NWBHDF5IO  # noqa: E402
from aind_dynamic_foraging_models.generative_model import ForagerCollection  # noqa: E402


# -----------------------------
# Forager spec
# -----------------------------
@dataclass(frozen=True)
class ForagerSpec:
    """Specification for a forager model to fit."""

    name: str
    preset: Optional[str] = None
    agent_alias: Optional[str] = None
    agent_class_name: Optional[str] = None
    agent_kwargs: Optional[Dict[str, Any]] = None
    clamp_params: Optional[Dict[str, Any]] = None


# -----------------------------
# NWB helpers
# -----------------------------
def get_nwb_from_path(nwb_path: str):
    """Open NWB file from a full path."""
    io = NWBHDF5IO(nwb_path, mode="r")
    nwb = io.read()
    return io, nwb


def get_protocol_from_nwb(nwb) -> Any:
    """Best-effort extraction of nwb.protocol."""
    try:
        return getattr(nwb, "protocol", None)
    except Exception:
        return None


def get_history_from_nwb(nwb) -> Tuple[bool, np.ndarray, np.ndarray, np.ndarray]:
    """
    Get choice and reward history from NWB file.

    Returns
    -------
    baiting : bool
    choice_history : np.ndarray of shape (n_trials,), values in {0,1} with NaNs for invalid choices
    reward_history : np.ndarray of shape (n_trials,), dtype bool/int
    autowater_offered : np.ndarray of shape (n_trials,), dtype bool
    """
    df_trial = nwb.trials.to_dataframe()

    autowater_offered = (df_trial.auto_waterL == 1) | (df_trial.auto_waterR == 1)
    choice_history = df_trial.animal_response.map({0: 0, 1: 1, 2: np.nan}).values
    reward_history = (df_trial.rewarded_historyL | df_trial.rewarded_historyR).to_numpy()

    baiting = False if "without baiting" in (nwb.protocol or "").lower() else True

    return baiting, choice_history, reward_history, autowater_offered.to_numpy()


def get_auto_train_stage_last(nwb) -> Any:
    """
    Extract the last auto_train_stage value from NWB trials, if present.
    Returns None if missing or any error occurs.
    """
    try:
        if hasattr(nwb, "trials") and "auto_train_stage" in nwb.trials.colnames:
            col = nwb.trials["auto_train_stage"].data[:]
            if len(col) > 0:
                return col[-1]
    except Exception:
        return None
    return None


# -----------------------------
# Forager builders
# -----------------------------
@lru_cache(maxsize=1)
def get_all_foragers_table():
    """
    Cache the full table returned by ForagerCollection.get_all_foragers().
    """
    fc = ForagerCollection()
    return fc.get_all_foragers()


@lru_cache(maxsize=1)
def get_forager_alias_map() -> Dict[str, Any]:
    """
    Build a mapping from agent_alias to forager object using
    ForagerCollection.get_all_foragers().

    Expected columns:
    - agent_alias
    - forager
    """
    df = get_all_foragers_table()

    required_cols = {"agent_alias", "forager"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise KeyError(
            "ForagerCollection.get_all_foragers() is missing required columns: "
            f"{sorted(missing_cols)}"
        )

    alias_map: Dict[str, Any] = {}
    for _, row in df.iterrows():
        alias = row["agent_alias"]
        forager = row["forager"]

        if alias is None:
            continue

        alias_map[str(alias)] = forager

    return alias_map


def validate_forager_spec(spec: ForagerSpec) -> None:
    """
    Ensure exactly one forager construction mode is provided.
    """
    n_sources = sum(
        x is not None
        for x in [spec.preset, spec.agent_alias, spec.agent_class_name]
    )
    if n_sources != 1:
        raise ValueError(
            "Each ForagerSpec must provide exactly one of: "
            "preset, agent_alias, or agent_class_name. "
            f"Got spec={spec}"
        )


def build_forager(spec: ForagerSpec):
    """
    Instantiate a forager from one of the following sources:
    1. preset
    2. agent_alias
    3. agent_class_name + agent_kwargs
    """
    validate_forager_spec(spec)
    fc = ForagerCollection()

    if spec.preset is not None:
        return fc.get_preset_forager(spec.preset)

    if spec.agent_alias is not None:
        alias_map = get_forager_alias_map()
        if spec.agent_alias not in alias_map:
            raise KeyError(
                f"agent_alias={spec.agent_alias!r} not found in "
                "ForagerCollection.get_all_foragers()['agent_alias']"
            )
        return copy.deepcopy(alias_map[spec.agent_alias])

    if spec.agent_class_name is not None:
        return fc.get_forager(
            agent_class_name=spec.agent_class_name,
            agent_kwargs=spec.agent_kwargs or {},
        )

    raise ValueError(f"Invalid ForagerSpec: {spec}")


def build_fit_bounds_for_forager(
    forager: Any,
    *,
    bounds_default: Dict[str, Tuple[float, float]],
) -> Dict[str, Tuple[float, float]]:
    """
    Build a fit-bounds dict that only includes keys that exist in the forager's
    ParamFitBoundModel. This avoids 'extra_forbidden' errors.
    """
    model_fields = getattr(forager.ParamFitBoundModel, "model_fields", None)
    if isinstance(model_fields, dict):
        allowed = set(model_fields.keys())
    else:
        allowed = set(dir(forager.ParamFitBoundModel))

    return {k: bounds_default[k] for k in bounds_default if k in allowed}


def fit_with_bounds(
    forager: Any,
    choice_valid: np.ndarray,
    reward_valid: np.ndarray,
    *,
    clamp_params: Dict[str, Any],
    de_kwargs: Dict[str, Any],
    fit_bounds: Dict[str, Tuple[float, float]],
):
    """
    Call forager.fit with fit-bounds override.
    """
    kw_candidates = [
        "fit_bounds_override",
        "fit_bounds",
        "fit_bounds_dict",
        "fit_bounds_override_dict",
    ]

    last_err: Optional[Exception] = None
    for k in kw_candidates:
        try:
            return forager.fit(
                choice_valid,
                reward_valid,
                clamp_params=clamp_params,
                DE_kwargs=de_kwargs,
                **{k: fit_bounds},
            )
        except TypeError as e:
            last_err = e
            continue

    raise TypeError(
        "Could not pass fit bounds into forager.fit(). Tried kwarg names: "
        f"{kw_candidates}. Last error: {type(last_err).__name__}: {last_err}"
    )


# -----------------------------
# Serialization helpers
# -----------------------------
def _to_jsonable(x: Any) -> Any:
    """Best-effort conversion to JSON-serializable types."""
    if isinstance(x, (str, int, float, bool)) or x is None:
        return x
    if isinstance(x, (list, tuple)):
        return [_to_jsonable(v) for v in x]
    if isinstance(x, dict):
        return {str(k): _to_jsonable(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.integer, np.floating, np.bool_)):
        return x.item()
    return str(x)


def save_json(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(_to_jsonable(payload), f, indent=2)


def load_json(path: Path) -> Dict[str, Any]:
    """Load JSON as dict."""
    with path.open("r") as f:
        return json.load(f)


def dump_fitted_agent(model_folder: Path, forager: Any) -> Dict[str, Any]:
    """
    Save the entire fitted agent.
    - First try stdlib pickle
    - Then fall back to cloudpickle
    - Then fall back to joblib
    """
    model_folder.mkdir(parents=True, exist_ok=True)

    try:
        import pickle

        p = model_folder / "fitted_agent.pkl"
        with p.open("wb") as f:
            pickle.dump(forager, f, protocol=pickle.HIGHEST_PROTOCOL)
        return {"pickle_saved": True, "pickle_backend": "pickle", "pickle_file": p.name}
    except Exception as e1:
        err1 = f"{type(e1).__name__}: {e1}"

    try:
        import cloudpickle  # type: ignore

        p = model_folder / "fitted_agent.cloudpickle"
        with p.open("wb") as f:
            cloudpickle.dump(forager, f)
        return {
            "pickle_saved": True,
            "pickle_backend": "cloudpickle",
            "pickle_file": p.name,
            "pickle_fallback_error": err1,
        }
    except Exception as e2:
        err2 = f"{type(e2).__name__}: {e2}"

    try:
        import joblib  # type: ignore

        p = model_folder / "fitted_agent.joblib"
        joblib.dump(forager, p, compress=3)
        return {
            "pickle_saved": True,
            "pickle_backend": "joblib",
            "pickle_file": p.name,
            "pickle_fallback_error": err1,
            "pickle_cloudpickle_error": err2,
        }
    except Exception as e3:
        err3 = f"{type(e3).__name__}: {e3}"

    return {
        "pickle_saved": False,
        "pickle_error": err1,
        "cloudpickle_error": err2,
        "joblib_error": err3,
    }


def get_model_result_json_path(session_out_dir: Path, model_name: str) -> Path:
    """Return the canonical JSON result path for one model."""
    return session_out_dir / model_name / f"{model_name}.json"


def all_model_result_jsons_exist(session_out_dir: Path, foragers: List[ForagerSpec]) -> bool:
    """Return True if every model already has its result JSON."""
    return all(get_model_result_json_path(session_out_dir, spec.name).exists() for spec in foragers)


# -----------------------------
# Core fit routine
# -----------------------------
def fit_one_session(
    session_path: str,
    *,
    results_root: Path,
    min_valid_trials: int,
    allowed_auto_train_stages: Optional[Set[str]],
    fit_bounds_default: Dict[str, Tuple[float, float]],
    de_kwargs: Dict[str, Any],
    save_figs: bool,
    fig_dpi: int,
    skip_existing_model_results: bool,
    skip_session_if_all_models_exist: bool,
    foragers: List[ForagerSpec],
) -> Dict[str, Any]:
    """
    Fit all foragers for a single NWB session.

    If allowed_auto_train_stages is None or empty, all sessions are fitted.
    Otherwise, only sessions whose last auto_train_stage is in the provided set
    will be fitted.
    """
    session_id = Path(session_path).stem
    print(f"[SESSION START] {session_id}", flush=True)

    session_summary: Dict[str, Any] = {
        "session_id": session_id,
        "nwb_path": session_path,
        "status": "ok",
        "n_valid_trials": None,
        "models": {},
    }

    io = None
    try:
        io, nwb = get_nwb_from_path(session_path)

        protocol = get_protocol_from_nwb(nwb)
        session_summary["protocol"] = None if protocol is None else _to_jsonable(protocol)

        auto_train_stage_last = get_auto_train_stage_last(nwb)
        stage_str = None if auto_train_stage_last is None else str(auto_train_stage_last)
        session_summary["auto_train_stage"] = stage_str

        stage_filter_enabled = bool(allowed_auto_train_stages)

        if stage_filter_enabled and stage_str not in allowed_auto_train_stages:
            session_summary["status"] = (
                f"skipped: auto_train_stage not in {sorted(allowed_auto_train_stages)}"
            )
            print(
                f"[{session_id}] SKIP session because auto_train_stage={stage_str} "
                f"is not in {sorted(allowed_auto_train_stages)}",
                flush=True,
            )
            return session_summary

        baiting, choice_history, reward_history, _autowater_offered = get_history_from_nwb(nwb)

        ignored = np.isnan(choice_history)
        choice_valid = choice_history[~ignored].astype(int)
        reward_valid = reward_history[~ignored].astype(int)

        session_summary["n_valid_trials"] = int(len(choice_valid))
        session_summary["baiting"] = bool(baiting)

        if len(choice_valid) < min_valid_trials:
            session_summary["status"] = f"skipped: valid trials < {min_valid_trials}"
            print(
                f"[{session_id}] SKIP session because valid trials={len(choice_valid)} "
                f"< {min_valid_trials}",
                flush=True,
            )
            return session_summary

        out_dir = results_root / session_id

        if skip_session_if_all_models_exist and out_dir.exists() and all_model_result_jsons_exist(out_dir, foragers):
            session_summary["status"] = "skipped: all model result JSON files already exist"
            print(
                f"[{session_id}] SKIP entire session because all model result JSON files already exist "
                f"under {out_dir}",
                flush=True,
            )

            for spec in foragers:
                result_json = get_model_result_json_path(out_dir, spec.name)
                try:
                    model_out = load_json(result_json)
                except Exception as e:
                    model_out = {
                        "status": "skipped_existing_but_unreadable",
                        "error": f"{type(e).__name__}: {e}",
                        "result_json": str(result_json),
                    }
                session_summary["models"][spec.name] = model_out

            return session_summary

        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"[{session_id}] Output folder: {out_dir}", flush=True)

        save_json(out_dir / "summary.json", session_summary)

        for spec in foragers:
            model_key = spec.name
            model_folder = out_dir / model_key
            result_json = model_folder / f"{model_key}.json"

            if skip_existing_model_results and result_json.exists():
                print(
                    f"[{session_id}] -> SKIP model: {model_key} "
                    f"(existing result found at {result_json})",
                    flush=True,
                )
                try:
                    model_out = load_json(result_json)
                    model_out["status"] = model_out.get("status", "skipped_existing")
                    model_out["skipped_existing"] = True
                    model_out["result_json"] = str(result_json)
                except Exception as e:
                    model_out = {
                        "status": "skipped_existing_but_unreadable",
                        "error": f"{type(e).__name__}: {e}",
                        "skipped_existing": True,
                        "result_json": str(result_json),
                    }

                session_summary["models"][model_key] = model_out
                continue

            print(f"[{session_id}] -> Fitting model: {model_key}", flush=True)

            model_out: Dict[str, Any] = {
                "status": "ok",
                "protocol": session_summary.get("protocol", None),
                "auto_train_stage": session_summary.get("auto_train_stage", None),
                "skipped_existing": False,
                "forager_spec": {
                    "name": spec.name,
                    "preset": spec.preset,
                    "agent_alias": spec.agent_alias,
                    "agent_class_name": spec.agent_class_name,
                    "agent_kwargs": _to_jsonable(spec.agent_kwargs),
                    "clamp_params": _to_jsonable(spec.clamp_params),
                },
            }

            try:
                forager = build_forager(spec)

                fit_bounds = build_fit_bounds_for_forager(
                    forager,
                    bounds_default=fit_bounds_default,
                )
                model_out["fit_bounds"] = _to_jsonable(fit_bounds)

                fitting_result, _ = fit_with_bounds(
                    forager,
                    choice_valid,
                    reward_valid,
                    clamp_params=spec.clamp_params or {},
                    de_kwargs=de_kwargs,
                    fit_bounds=fit_bounds,
                )

                model_folder.mkdir(parents=True, exist_ok=True)
                model_out.update(dump_fitted_agent(model_folder, forager))

                fit_settings = getattr(fitting_result, "fit_settings", {}) or {}
                fit_names = fit_settings.get("fit_names", None)

                model_out.update(
                    dict(
                        n_trials=int(len(choice_valid)),
                        LPT=float(getattr(fitting_result, "LPT", np.nan)),
                        AIC=float(getattr(fitting_result, "AIC", np.nan)),
                        BIC=float(getattr(fitting_result, "BIC", np.nan)),
                        prediction_accuracy=float(getattr(fitting_result, "prediction_accuracy", np.nan)),
                        fit_names=fit_names,
                        x=[float(v) for v in list(getattr(fitting_result, "x", []))],
                    )
                )

                if save_figs:
                    try:
                        fig, _axes = forager.plot_fitted_session(if_plot_latent=True)
                        fig.savefig(model_folder / f"{model_key}.png", dpi=fig_dpi)
                        plt.close(fig)
                        model_out["fig_saved"] = True
                    except Exception as e:
                        model_out["fig_saved"] = False
                        model_out["fig_error"] = f"{type(e).__name__}: {e}"

                save_json(result_json, model_out)

            except Exception as e:
                model_out["status"] = "error"
                model_out["error"] = f"{type(e).__name__}: {e}"
                model_out["traceback"] = traceback.format_exc()
                save_json(result_json, model_out)

            session_summary["models"][model_key] = model_out

        save_json(out_dir / "summary.json", session_summary)
        print(f"[SESSION DONE] {session_id}", flush=True)
        return session_summary

    except Exception as e:
        session_summary["status"] = "error"
        session_summary["error"] = f"{type(e).__name__}: {e}"
        session_summary["traceback"] = traceback.format_exc()
        print(f"[SESSION ERROR] {session_id}: {type(e).__name__}: {e}", flush=True)
        return session_summary

    finally:
        try:
            if io is not None:
                io.close()
        except Exception:
            pass


# -----------------------------
# Session discovery
# -----------------------------
def find_all_sessions(local_roots: List[str]) -> List[str]:
    """Return all .nwb file paths across all roots, deduplicated."""
    all_paths: List[str] = []
    for root in local_roots:
        all_paths.extend(glob.glob(f"{root}/*.nwb"))
    return sorted(set(all_paths), key=os.path.getsize, reverse=True)

from concurrent.futures import ProcessPoolExecutor, as_completed

def main(
    *,
    local_nwb_roots: List[str],
    results_root: Path,
    min_valid_trials: int,
    allowed_auto_train_stages: Optional[Set[str]] = None,
    de_kwargs: Dict[str, Any],
    save_figs: bool,
    fig_dpi: int,
    max_workers: int,
    skip_existing_model_results: bool,
    skip_session_if_all_models_exist: bool,
    fit_bounds_default: Dict[str, Tuple[float, float]],
    foragers: List[ForagerSpec],
) -> None:
    results_root.mkdir(parents=True, exist_ok=True)

    session_paths = find_all_sessions(local_nwb_roots)
    if len(session_paths) == 0:
        raise RuntimeError(f"No .nwb files found under: {local_nwb_roots}")

    total_jobs = len(session_paths)
    finished_jobs = 0

    print(f"[MAIN] Found {total_jobs} session(s)", flush=True)
    print(f"[MAIN] RESULTS_ROOT = {results_root}", flush=True)

    all_summaries: List[Dict[str, Any]] = []

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(
                fit_one_session,
                p,
                results_root=results_root,
                min_valid_trials=min_valid_trials,
                allowed_auto_train_stages=allowed_auto_train_stages,
                fit_bounds_default=fit_bounds_default,
                de_kwargs=de_kwargs,
                save_figs=save_figs,
                fig_dpi=fig_dpi,
                skip_existing_model_results=skip_existing_model_results,
                skip_session_if_all_models_exist=skip_session_if_all_models_exist,
                foragers=foragers,
            ): p
            for p in session_paths
        }

        for fut in as_completed(futures):
            session_path = futures[fut]
            session_id = Path(session_path).stem
            try:
                summary = fut.result()
                all_summaries.append(summary)
            except Exception as e:
                summary = {
                    "session_id": session_id,
                    "status": "error",
                    "error": f"{type(e).__name__}: {e}",
                }
                all_summaries.append(summary)

            finished_jobs += 1
            print(f"[PROGRESS] {finished_jobs}/{total_jobs} sessions finished", flush=True)

    # Save global summary as before
    global_summary = {
        "local_roots": local_nwb_roots,
        "results_root": str(results_root),
        "n_sessions_found": total_jobs,
        "n_sessions_processed": finished_jobs,
        "foragers": [
            {
                "name": spec.name,
                "preset": spec.preset,
                "agent_alias": spec.agent_alias,
                "agent_class_name": spec.agent_class_name,
                "agent_kwargs": _to_jsonable(spec.agent_kwargs),
                "clamp_params": _to_jsonable(spec.clamp_params),
            }
            for spec in foragers
        ],
        "summaries": all_summaries,
        "fit_bounds_default": _to_jsonable(fit_bounds_default),
        "allowed_auto_train_stages": (
            sorted(allowed_auto_train_stages) if allowed_auto_train_stages else None
        ),
        "skip_existing_model_results": skip_existing_model_results,
        "skip_session_if_all_models_exist": skip_session_if_all_models_exist,
    }

    save_json(results_root / "ALL_SESSIONS_SUMMARY.json", global_summary)
    print(f"[MAIN] Saved global summary to {results_root / 'ALL_SESSIONS_SUMMARY.json'}", flush=True)




In [2]:
import psutil
import os
import signal

# Get the current notebook's PID
current_pid = os.getpid()
print(f"Current notebook PID: {current_pid}")

# Iterate through all processes
for proc in psutil.process_iter(['pid', 'name', 'cpu_percent', 'cmdline']):
    try:
        pid = proc.info['pid']
        name = proc.info['name']
        cmdline = proc.info['cmdline']

        # Skip current notebook process
        if pid == current_pid:
            continue

        # Heuristic: look for Python processes started by Jupyter/your notebook
        if 'python' in name.lower() and cmdline:
            # Optional: filter only processes that look like your previous workers
            # Example: if using ProcessPoolExecutor, cmdline may include 'concurrent' or the script path
            if 'concurrent.futures' in ' '.join(cmdline) or 'ipykernel_launcher' in ' '.join(cmdline):
                print(f"Killing PID {pid} - {cmdline}")
                proc.kill()  # or proc.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        continue

print("Finished cleaning leftover Python processes.")

# -----------------------------
# User config
# -----------------------------
LOCAL_NWB_ROOTS: List[str] = [
    "/root/capsule/data/behavior_nwb",
    "/data/foraging_nwb_bonsai",
    # "/data/other_folder",
]

RESULTS_ROOT = Path("/root/capsule/scratch/model_comparison")
MIN_VALID_TRIALS = 400

# Only fit sessions whose LAST auto_train_stage is in this allow-list
ALLOWED_AUTO_TRAIN_STAGES = {"GRADUATED"}
#ALLOWED_AUTO_TRAIN_STAGES=None # for all sessions
# Differential evolution config
DE_KWARGS = dict(
    workers=4,
    disp=False,
    seed=np.random.default_rng(42),
)

# If True, save fitted-session plots
SAVE_FIGS = False
FIG_DPI = 150

# Parallelism across sessions
MAX_WORKERS = max(1, (os.cpu_count() or 2) // 2)
MAX_WORKERS = 120

# If True, skip a model when its final JSON result already exists
SKIP_EXISTING_MODEL_RESULTS = True

# If True, skip an entire session when all model result JSON files already exist
SKIP_SESSION_IF_ALL_MODELS_EXIST = True

# -----------------------------
# Fit bounds
# -----------------------------
FIT_BOUNDS_DEFAULT: Dict[str, Tuple[float, float]] = {
    "threshold": (-10, 10),
    "biasL": (-10, 10),
    "stay_bias": (-10, 10),
}

# -----------------------------
# Models to fit
# -----------------------------
FORAGERS: List[ForagerSpec] = [
    # -------------------------
    # L1, FixThrT0
    # -------------------------
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetT_StayBiasT_SideBiasF_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": True,  "include_stay_bias": True,  "include_side_bias": False, "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetT_StayBiasT_SideBiasT_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": True,  "include_stay_bias": True,  "include_side_bias": True,  "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetT_StayBiasF_SideBiasF_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": True,  "include_stay_bias": False, "include_side_bias": False, "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetT_StayBiasF_SideBiasT_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": True,  "include_stay_bias": False, "include_side_bias": True,  "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetF_StayBiasT_SideBiasF_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": False, "include_stay_bias": True,  "include_side_bias": False, "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetF_StayBiasT_SideBiasT_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": False, "include_stay_bias": True,  "include_side_bias": True,  "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetF_StayBiasF_SideBiasF_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": False, "include_stay_bias": False, "include_side_bias": False, "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetF_StayBiasF_SideBiasT_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": False, "include_stay_bias": False, "include_side_bias": True,  "fix_threshold": True,  "threshold_fixed": 0}),

    # -------------------------
    # L1, FixThrF
    # -------------------------
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetT_StayBiasT_SideBiasF_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": True,  "include_stay_bias": True,  "include_side_bias": False, "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetT_StayBiasT_SideBiasT_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": True,  "include_stay_bias": True,  "include_side_bias": True,  "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetT_StayBiasF_SideBiasF_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": True,  "include_stay_bias": False, "include_side_bias": False, "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetT_StayBiasF_SideBiasT_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": True,  "include_stay_bias": False, "include_side_bias": True,  "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetF_StayBiasT_SideBiasF_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": False, "include_stay_bias": True,  "include_side_bias": False, "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetF_StayBiasT_SideBiasT_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": False, "include_stay_bias": True,  "include_side_bias": True,  "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetF_StayBiasF_SideBiasF_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": False, "include_stay_bias": False, "include_side_bias": False, "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L1_ResetF_StayBiasF_SideBiasT_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": False, "include_stay_bias": False, "include_side_bias": True,  "fix_threshold": False, "threshold_fixed": 0}),

    # -------------------------
    # L2, FixThrT0
    # -------------------------
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetT_StayBiasT_SideBiasF_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": True,  "include_stay_bias": True,  "include_side_bias": False, "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetT_StayBiasT_SideBiasT_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": True,  "include_stay_bias": True,  "include_side_bias": True,  "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetT_StayBiasF_SideBiasF_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": True,  "include_stay_bias": False, "include_side_bias": False, "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetT_StayBiasF_SideBiasT_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": True,  "include_stay_bias": False, "include_side_bias": True,  "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetF_StayBiasT_SideBiasF_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": False, "include_stay_bias": True,  "include_side_bias": False, "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetF_StayBiasT_SideBiasT_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": False, "include_stay_bias": True,  "include_side_bias": True,  "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetF_StayBiasF_SideBiasF_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": False, "include_stay_bias": False, "include_side_bias": False, "fix_threshold": True,  "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetF_StayBiasF_SideBiasT_FixThrT0", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": False, "include_stay_bias": False, "include_side_bias": True,  "fix_threshold": True,  "threshold_fixed": 0}),

    # -------------------------
    # L2, FixThrF
    # -------------------------
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetT_StayBiasT_SideBiasF_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": True,  "include_stay_bias": True,  "include_side_bias": False, "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetT_StayBiasT_SideBiasT_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": True,  "include_stay_bias": True,  "include_side_bias": True,  "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetT_StayBiasF_SideBiasF_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": True,  "include_stay_bias": False, "include_side_bias": False, "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetT_StayBiasF_SideBiasT_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": True,  "include_stay_bias": False, "include_side_bias": True,  "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetF_StayBiasT_SideBiasF_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": False, "include_stay_bias": True,  "include_side_bias": False, "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetF_StayBiasT_SideBiasT_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": False, "include_stay_bias": True,  "include_side_bias": True,  "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetF_StayBiasF_SideBiasF_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": False, "include_stay_bias": False, "include_side_bias": False, "fix_threshold": False, "threshold_fixed": 0}),
    ForagerSpec(name="ForagingCompareThreshold_L2_ResetF_StayBiasF_SideBiasT_FixThrF", agent_class_name="ForagerCompareThreshold", agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": False, "include_stay_bias": False, "include_side_bias": True,  "fix_threshold": False, "threshold_fixed": 0}),

    # preset
    ForagerSpec(name="Hattori2019", preset="Hattori2019"),
    ForagerSpec(name="Bari2019", preset="Bari2019"),

    # Alias-based models
    ForagerSpec(name="QLearning_L1F0_softmax", agent_alias="QLearning_L1F0_softmax"),
    ForagerSpec(name="QLearning_L1F0_epsi", agent_alias="QLearning_L1F0_epsi"),
    ForagerSpec(name="QLearning_L1F0_CK1_softmax", agent_alias="QLearning_L1F0_CK1_softmax"),
    ForagerSpec(name="QLearning_L1F0_CK1_epsi", agent_alias="QLearning_L1F0_CK1_epsi"),
    ForagerSpec(name="QLearning_L1F0_CKfull_softmax", agent_alias="QLearning_L1F0_CKfull_softmax"),
    ForagerSpec(name="QLearning_L1F0_CKfull_epsi", agent_alias="QLearning_L1F0_CKfull_epsi"),
    ForagerSpec(name="QLearning_L1F1_softmax", agent_alias="QLearning_L1F1_softmax"),
    ForagerSpec(name="QLearning_L1F1_epsi", agent_alias="QLearning_L1F1_epsi"),
    ForagerSpec(name="QLearning_L1F1_CK1_softmax", agent_alias="QLearning_L1F1_CK1_softmax"),
    ForagerSpec(name="QLearning_L1F1_CK1_epsi", agent_alias="QLearning_L1F1_CK1_epsi"),
    ForagerSpec(name="QLearning_L1F1_CKfull_softmax", agent_alias="QLearning_L1F1_CKfull_softmax"),
    ForagerSpec(name="QLearning_L1F1_CKfull_epsi", agent_alias="QLearning_L1F1_CKfull_epsi"),
    ForagerSpec(name="QLearning_L2F0_softmax", agent_alias="QLearning_L2F0_softmax"),
    ForagerSpec(name="QLearning_L2F0_epsi", agent_alias="QLearning_L2F0_epsi"),
    ForagerSpec(name="QLearning_L2F0_CK1_softmax", agent_alias="QLearning_L2F0_CK1_softmax"),
    ForagerSpec(name="QLearning_L2F0_CK1_epsi", agent_alias="QLearning_L2F0_CK1_epsi"),
    ForagerSpec(name="QLearning_L2F0_CKfull_softmax", agent_alias="QLearning_L2F0_CKfull_softmax"),
    ForagerSpec(name="QLearning_L2F0_CKfull_epsi", agent_alias="QLearning_L2F0_CKfull_epsi"),
    ForagerSpec(name="QLearning_L2F1_softmax", agent_alias="QLearning_L2F1_softmax"),
    ForagerSpec(name="QLearning_L2F1_epsi", agent_alias="QLearning_L2F1_epsi"),
    ForagerSpec(name="QLearning_L2F1_CK1_softmax", agent_alias="QLearning_L2F1_CK1_softmax"),
    ForagerSpec(name="QLearning_L2F1_CK1_epsi", agent_alias="QLearning_L2F1_CK1_epsi"),
    ForagerSpec(name="QLearning_L2F1_CKfull_softmax", agent_alias="QLearning_L2F1_CKfull_softmax"),
    ForagerSpec(name="QLearning_L2F1_CKfull_epsi", agent_alias="QLearning_L2F1_CKfull_epsi"),
    ForagerSpec(name="LossCounting", agent_alias="LossCounting"),
    ForagerSpec(name="LossCounting_CK1", agent_alias="LossCounting_CK1"),
    ForagerSpec(name="LossCounting_CKfull", agent_alias="LossCounting_CKfull"),
    ForagerSpec(name="WSLS", agent_alias="WSLS"),
    ForagerSpec(name="WSLS_CK1", agent_alias="WSLS_CK1"),
    ForagerSpec(name="WSLS_CKfull", agent_alias="WSLS_CKfull"),

    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasT_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasT_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasT_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasT_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasT_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasT_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasT_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasT_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasF_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasF_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasF_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasF_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasF_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasF_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasF_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L1_CK1_ResetT_StayBiasF_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasT_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasT_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasT_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasT_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasT_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasT_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasT_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasT_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasF_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasF_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasF_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasF_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasF_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasF_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasF_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L1_CK1_ResetF_StayBiasF_SideBiasF_FixThrF"),

    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasT_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasT_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasT_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasT_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasT_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasT_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasT_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasT_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasF_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasF_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasF_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasF_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasF_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasF_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasF_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetT_StayBiasF_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasT_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasT_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasT_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasT_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasT_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasT_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasT_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasT_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasF_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasF_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasF_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasF_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasF_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasF_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasF_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasF_SideBiasF_FixThrF"),

    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasT_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasT_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasT_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasT_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasT_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasT_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasT_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasT_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasF_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasF_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasF_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasF_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasF_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasF_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasF_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L2_CK1_ResetT_StayBiasF_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasT_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasT_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasT_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasT_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasT_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasT_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasT_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasT_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasF_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasF_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasF_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasF_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasF_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasF_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasF_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L2_CK1_ResetF_StayBiasF_SideBiasF_FixThrF"),

    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasT_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasT_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasT_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasT_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasT_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasT_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasT_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasT_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasF_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasF_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasF_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasF_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasF_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasF_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasF_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetT_StayBiasF_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasT_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasT_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasT_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasT_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasT_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasT_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasT_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasT_SideBiasF_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasF_SideBiasT_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasF_SideBiasT_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasF_SideBiasT_FixThrF", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasF_SideBiasT_FixThrF"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasF_SideBiasF_FixThrT0.00", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasF_SideBiasF_FixThrT0.00"),
    ForagerSpec(name="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasF_SideBiasF_FixThrF", agent_alias="ForagingCompareThreshold_L2_CKfull_ResetF_StayBiasF_SideBiasF_FixThrF"),
]

# -----------------------------
# Start
# -----------------------------
main(
    local_nwb_roots=LOCAL_NWB_ROOTS,
    results_root=RESULTS_ROOT,
    min_valid_trials=MIN_VALID_TRIALS,
    allowed_auto_train_stages=ALLOWED_AUTO_TRAIN_STAGES,
    de_kwargs=DE_KWARGS,
    save_figs=SAVE_FIGS,
    fig_dpi=FIG_DPI,
    max_workers=MAX_WORKERS,
    skip_existing_model_results=SKIP_EXISTING_MODEL_RESULTS,
    skip_session_if_all_models_exist=SKIP_SESSION_IF_ALL_MODELS_EXIST,
    fit_bounds_default=FIT_BOUNDS_DEFAULT,
    foragers=FORAGERS,
)


[779310_2025-05-08_13-26-26] SKIP session because auto_train_stage=STAGE_3 is not in ['GRADUATED']
[PROGRESS] 6235/19148 sessions finished
[SESSION START] 662914_2023-09-25


/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:620: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.4.0, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


[662914_2023-09-25] SKIP session because auto_train_stage=None is not in ['GRADUATED']
[PROGRESS] 6236/19148 sessions finished
[SESSION START] 791096_2025-06-02_13-30-22
[791691_2025-06-05_11-07-24] -> Fitting model: ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasT_SideBiasF_FixThrT0.00


2026-03-13 04:50:49,806 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...


[754897_2025-03-03_09-01-25] -> Fitting model: ForagingCompareThreshold_L1_CK1_ResetF_StayBiasT_SideBiasF_FixThrF


2026-03-13 04:50:50,531 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...


[791096_2025-06-02_13-30-22] SKIP session because auto_train_stage=STAGE_2 is not in ['GRADUATED']
[PROGRESS] 6237/19148 sessions finished
[SESSION START] 750108_2024-09-26_09-11-57
[791691_2025-05-28_11-00-54] -> Fitting model: ForagingCompareThreshold_L1_CKfull_ResetF_StayBiasF_SideBiasT_FixThrT0.00


2026-03-13 04:50:53,266 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...


[733683_2024-10-09_11-02-02] -> Fitting model: ForagingCompareThreshold_L1_CK1_ResetT_StayBiasF_SideBiasF_FixThrT0.00


2026-03-13 04:50:53,746 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
